In [ ]:
%%capture
%pip install QuantumRingsLib numpy matplotlib colorama


In [ ]:
#
# Setup your account
# You can also save your account locally using the class method QrRuntimeService.save_account(...) and
# invoke the QrRuntimeService class constructor without any arguments.
#

import os

my_token = os.environ["QR_TOKEN"]
my_name = os.environ["QR_ACCOUNT"]

#
# Set the backend of your choice, depending upon the task and your hardware configuration.
# See SDK documentation for additional help.
#

my_backend = "amber_quantum_rings"
precision = "double"

In [ ]:
from platform import python_version
print(python_version())
import os
import sys
import platform
from os import listdir
from os.path import isfile, join
from colorama import Fore, Style, init
from itertools import islice
from collections import namedtuple

if platform.system() == "Windows":
    cuda_path = os.getenv("CUDA_PATH", "")
    if "" == cuda_path:
        #set a hard-coded path
        cuda_path = "C:\\Program Files\\NVIDIA GPU Computing Toolkit\\CUDA\\v13.0\\bin\\x64"
    else:
        #create from the environment
        if "13" in cuda_path:
            cuda_path += "\\bin\\x64"
        else:
            cuda_path += "\\bin"
            
    os.add_dll_directory(cuda_path)


import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from QuantumRingsLib import OptimizeQuantumCircuit
from matplotlib import pyplot as plt
import numpy as np
import math
import time
from collections import Counter
import json
import csv
from enum import Enum

provider = QuantumRingsProvider(token=my_token, name=my_name)
backend = provider.get_backend(my_backend, precision = precision)
print("Account Name: ", provider.active_account()["name"], "\nMax Qubits: ", provider.active_account()["max_qubits"])

In [ ]:
number_of_shots = 1000
threshold = 16
num_qubits = 10

In [ ]:
# Build a Quantum Circuit
q = QuantumRegister(num_qubits)
c = ClassicalRegister(num_qubits)
qc = QuantumCircuit(q, c)

for i in range(num_qubits):
    qc.u(math.pi/2, -math.pi/2, -math.pi/4, q[i])

n = num_qubits
while (n):
    qc.h(q[n-1])
    for i in range (n-1, 0, -1):
        qc.cu1(math.pi / 2** (n - i ), q[i - 1], q[n-1])
    n -= 1

qc.measure_all();

In [ ]:
job = backend.run(
        qc, 
        shots=number_of_shots, 
        mode="async", 
        quiet=True, 
        performance="custom", 
        threshold=threshold
        )
job_monitor(job, quiet=True)
result = job.result()
print(f"Counts: {result.get_counts()}")
print(f"Peak Simulation Size: {result.get_peakmemorysize(0)}")
